# Local AI 800 — Google Colab training

Use a Colab runtime with a CUDA GPU. This notebook trains a LoRA adapter, merges it, and exports a Q4_K_M GGUF for the Android app. GPU availability and free-tier limits vary. Review dataset licenses before training or redistribution.

**Important:** this is a starter 0.6B model, not a guarantee of expert-level coding or reverse-engineering ability.

In [ ]:
# Check that Colab assigned a GPU
!nvidia-smi

In [ ]:
# Get the project
!git clone https://github.com/dragonbloxfruit01-stack/hhj.git
%cd /content/hhj
!pip install -q -r training/requirements.txt

In [ ]:
# Build starter dataset: public code + shell instruction data + original Chinese technical examples
!python training/prepare_public_data.py --output training/data/train.jsonl
!python training/validate_dataset.py training/data/train.jsonl

In [ ]:
# Train QLoRA. This can take time and GPU memory; adjust max-length/epochs if needed.
!python training/train_lora.py --model Qwen/Qwen3-0.6B --train-file training/data/train.jsonl --output-dir /content/localai800-lora --epochs 2 --max-length 2048

In [ ]:
# Merge LoRA adapter into the base model
!python training/merge_adapter.py --adapter /content/localai800-lora --output-dir /content/localai800-merged

In [ ]:
# Convert and quantize to GGUF
!git clone --depth 1 --branch b12182 https://github.com/ggml-org/llama.cpp.git
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S llama.cpp -B llama.cpp/build
!cmake --build llama.cpp/build --target llama-quantize -j2
!bash training/quantize_to_gguf.sh /content/localai800-merged /content/hhj/llama.cpp Q4_K_M

In [ ]:
# Download the trained model to your device
from google.colab import files
files.download('/content/localai800-merged/localai800-Q4_K_M.gguf')